# The question scoreboard


**What this reads (instruction 73, phase 4).** Every admitted catalogue question scored through the harness (Einar, 2026-11-08: keep all admitted): its own dataset family and information category held out, orthogroup-grouped nested cross-validation, the frozen test touched once. A question is one yes/no label: the extreme 20 % of a value column, or one class against the rest.

**How to read a row.** Frozen AUC 0.5 is chance; average precision is compared with the prevalence of yes; *may fill* means the frozen-test lower bound on precision reached 0.90 at some call threshold.

Built 2026-11-08.

In [1]:
import glob, json, os, numpy as np, pandas as pd
cat = pd.read_csv(os.path.join(REPO, 'results/question_catalogue_2026_11_08/catalogue.tsv'), sep='\t')
admitted = set(cat[cat.admitted].column)
rows = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(REPO, 'results/question_scores_2026_11_08/*.json')))]
S = pd.DataFrame(rows)
errors = S[S.get('error').notna()] if 'error' in S else S.iloc[:0]
S = S[S.column.isin(admitted) & (S.get('error').isna() if 'error' in S else True)]
print(len(rows), 'scored;', len(errors), 'errors;', len(S), 'on admitted columns')
S['lift_ap'] = (S.frozen_ap / S.prevalence).round(2)
S.to_csv(os.path.join(OUT, 'scores.tsv'), sep='\t', index=False)
errors.to_csv(os.path.join(OUT, 'errors.tsv'), sep='\t', index=False)

268 scored; 0 errors; 198 on admitted columns


## 1 · Per category

In [2]:
board = S.groupby('category').agg(questions=('question', 'size'), median_frozen_auc=('frozen_auc', 'median'),
                                  above_070=('frozen_auc', lambda a: int((a >= 0.70).sum())),
                                  near_chance=('frozen_auc', lambda a: int((a < 0.60).sum())),
                                  may_fill=('may_fill', 'sum')).round(3).sort_values('median_frozen_auc', ascending=False)
board.to_csv(os.path.join(OUT, 'by_category.tsv'), sep='\t')
summary = dict(questions=len(S), median_frozen_auc=round(float(S.frozen_auc.median()), 3),
               above_070=int((S.frozen_auc >= 0.70).sum()), near_chance=int((S.frozen_auc < 0.60).sum()),
               may_fill=int(S.may_fill.sum()), errors=len(errors))
json.dump(summary, open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
print(summary)
board

{'questions': 198, 'median_frozen_auc': 0.758, 'above_070': 134, 'near_chance': 30, 'may_fill': 7, 'errors': 0}


,questions,median_frozen_auc,above_070,near_chance,may_fill
category,,,,,
place,27,0.934,26,1,1
function,2,0.89,2,0,0
protein_abundance,7,0.843,7,0,0
transcript,67,0.764,56,1,2
translation,22,0.762,19,0,1
interaction,12,0.74,7,1,1
fitness,31,0.669,10,13,2
modification,22,0.661,6,8,0
host_response,2,0.593,0,1,0


## 2 · The best-answered and the worst-answered questions

In [3]:
cols = ['question', 'category', 'labelled', 'positives', 'frozen_auc', 'frozen_ap', 'prevalence', 'may_fill', 'top_families']
S.sort_values('frozen_auc', ascending=False)[cols].head(20)

,question,category,labelled,positives,frozen_auc,frozen_ap,prevalence,may_fill,top_families
216,cat_pvm_proximity_positive_1_0,place,1274,73,1,1,0.0573,False,"[gra12, interpro, unregistered]"
61,cat_compartment_rhoptries_1,place,8140,71,0.9993,0.8598,0.008722,False,"[gra12, gse19092, unregistered]"
56,cat_compartment_micronemes,place,8140,56,0.9989,0.925,0.00688,False,"[gse19092, iron_depletion_proteome, brady_subtypes]"
249,cat_te129869_confluent_r2_change,translation,7880,1650,0.9915,0.9703,0.2094,False,"[gse245775, gse168465, mrna_decay_gse329845]"
62,cat_compartment_rhoptries_2,place,8140,52,0.986,0.2146,0.006388,False,"[gra12, gse19092, gse302108_utr5]"
227,cat_rpf129869_confluent_r1_change,translation,7880,1855,0.9854,0.9523,0.2354,True,"[gse168465, gse245775, gse108740]"
42,cat_compartment_apical_1,place,8140,59,0.9788,0.4365,0.007248,False,"[gse19092, gse108740, iron_depletion_proteome]"
251,cat_te129869_subconfluent_r1_change,translation,7880,1704,0.9787,0.9314,0.2162,False,"[gse168465, gse245775, mrna_decay_gse329845]"
226,cat_rna99395_extracellular_r1_change,transcript,7506,1522,0.9744,0.8949,0.2028,False,"[gse129869, gse245775, unregistered]"
65,cat_compartment_unassigned,place,8140,4313,0.9741,0.9794,0.5299,False,"[iron_depletion_proteome, gse253884_5, proteome_total]"


In [4]:
S.sort_values('frozen_auc')[cols].head(15)

,question,category,labelled,positives,frozen_auc,frozen_ap,prevalence,may_fill,top_families
232,cat_screen_egress_phenotype_True,phenotype,318,35,0.3788,0.0659,0.1101,False,"[crispr_glucose_limitation, crispr_invivo_composite, gse245775]"
184,cat_n_o_fucosyl_peptides_6_0,modification,394,58,0.3929,0.1203,0.1472,False,"[starpath_xlms, lopit_tgon, gse99395]"
67,cat_crispr_gra12s1_disco_low,fitness,236,48,0.404,0.08829,0.2034,False,"[gse245775, gse19092, gse129869]"
164,cat_mrna_remaining_5h_actinomycin_high,transcript,412,83,0.4043,0.2275,0.2015,False,"[crispr_invivo_composite, gse99395, gse302107_riboseq]"
70,cat_crispr_gra12s2_disco_low,fitness,232,47,0.4222,0.19,0.2026,False,"[gse129869, iron_depletion_proteome, gse51780]"
231,cat_screen_apicoplast_phenotype_True,phenotype,318,60,0.4524,0.1367,0.1887,False,"[oxidative_stress_screen, gse99395, gse206344]"
181,cat_n_nitrosylation_sites_2_0,modification,660,136,0.4564,0.2021,0.2061,False,"[gse168465, interpro, proteome_total]"
211,cat_phospho_up_ratio_high,modification,1182,237,0.4672,0.1833,0.2005,False,"[crispr_serum_restriction, gse129869, gse108740]"
230,cat_screen_any_phenotype_True,phenotype,318,99,0.4676,0.2097,0.3113,False,"[gse51780, iron_depletion_proteome, crispr_serum_restriction]"
229,cat_screen_actin_phenotype_True,phenotype,318,44,0.5,0.2065,0.1384,False,"[gse129869, toxodb_enteroepithelial, alphafold]"


## 3 · Questions that may fill

Only these may write a cell into the map as a fact; every other question ranks genes and says how often such a ranking is right.

In [5]:
S[S.may_fill][cols]

,question,category,labelled,positives,frozen_auc,frozen_ap,prevalence,may_fill,top_families
73,cat_crispr_gra17_candidate_0_0,fitness,7553,7435,0.7978,0.9947,0.9844,True,"[gse108740, toxodb_enteroepithelial, brady_subtypes]"
83,cat_fit_density_dependence_log10padj_low,fitness,4173,4059,0.6693,0.9818,0.9727,True,"[gse108740, gse206344, invivo_brain_transcriptome]"
106,cat_invivo_WholeBrain_Acute_1_change,transcript,7663,3017,0.8743,0.8405,0.3937,True,"[gse245775, gse277553_cuttag, gse129869]"
157,cat_lopit_unified_nucleus,place,3827,1262,0.9141,0.8483,0.3298,True,"[unregistered, gse253884_5, iron_depletion_proteome]"
160,cat_macrophage_expression_percentile_high,transcript,8140,1634,0.9687,0.9183,0.2007,True,"[gse245775, unregistered, gse99395]"
173,cat_n_holes_0,interaction,8140,7683,0.7252,0.9747,0.9439,True,"[toxodb_enteroepithelial, gse108740, crispr_invitro]"
227,cat_rpf129869_confluent_r1_change,translation,7880,1855,0.9854,0.9523,0.2354,True,"[gse168465, gse245775, gse108740]"
